In [1]:
# Parameters
UF = "AC"


# Capítulo 3 — A Rede de Monitoramento Brasileira

Este notebook reproduz as figuras 20, 21 e 22 da seção 3 do Relatório Anual de Qualidade do Ar

LEIA ATENTAMENTE ANTES DE EXECUTAR ESTE  SCRIPT

1. ATUALIZAÇÃO DOS CAMINHOS:

   Este código precisa saber em qual pasta do seu computador o projeto está salvo.
   Como cada computador possui uma estrutura de pastas diferente, você deve 
   atualizar a variável de caminho bruto (quando indicado) para
   apontar para o diretório correto na sua máquina antes de rodar o script.


2. PRESERVAÇÃO DA ESTRUTURA DE PASTAS:

   Este código funciona de forma integrada com as demais pastas, scripts e
   arquivos auxiliares exatamente na ORGANIZAÇÃO fornecida no projeto.

   -> Não mova arquivos ou pastas de lugar.

   -> Não altere o nome dos diretórios ou arquivos.

   Caso a estrutura fornecida seja alterada, as importações e chamadas de dados 
   irão falhar e o código não irá funcionar.


### Fig. 20 — Número de estações de monitoramento por UF

Gráfico de barras empilhadas interativo que exibe, para cada ano,
a quantidade de estações de monitoramento ativas por Unidade Federativa (UF).

Botões no topo permitem filtrar por categoria de estação:  
**Todas · Referência · Indicativa · Não Declarado**.

In [2]:
# Importações necessárias
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import os
from IPython.display import HTML, display
from matplotlib import colormaps as m_colormaps
from matplotlib import colors as mcolors

# 1. Caminhos e Leitura do CSV
aqmData = pd.read_csv('https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv')
rootPath = os.path.dirname(os.getcwd())

# 1. Normalizar ID_OEMA: remover espaços para evitar duplicatas como "SP 001" vs "SP001"
aqmData = aqmData[aqmData["UF"] == UF]
aqmData["ID_OEMA"] = aqmData["ID_OEMA"].astype(str).str.replace(" ", "")
# 2. Agora sim removemos duplicadas
aqmData = aqmData.drop_duplicates(subset=["ID_OEMA"])

# 2.2. Parse dos anos: Converter a coluna de anos monitorados de string para lista de inteiros
def parse_anos(x):
    """Converte a representação textual de anos (ex.: "[2020, 2021]") em lista de int."""
    try:
        return [
            int(z) for z in str(x)
            .replace("[","").replace("]","").replace("'","").replace(" ","")
            .split(",") if z.strip().isdigit()
        ]
    except:
        return []

aqmData["ANOS_MONITORADOS"] = aqmData["ANOS_MONITORADOS"].apply(parse_anos)

# 2.3 Padronizar STATUS: preencher nulos e converter para minúsculas
if "STATUS" not in aqmData.columns:
    aqmData["STATUS"] = "Não declarado"
aqmData["STATUS"] = aqmData["STATUS"].fillna("Não declarado").astype(str).str.strip().str.lower()

# 2.4 Manter apenas estações ativas no ano vigente da análise
# Ex.: 2024 ou com 2024 na lista de anos monitorados
def has_year(lst, year=2024):
    return isinstance(lst, list) and (year in lst)

def is_active(status):
    return status == "ativa"

# Para analisar outros anos, basta alterar a função has_year para verificar o ano desejado.
aqmData["TEM_2024"] = aqmData["ANOS_MONITORADOS"].apply(lambda x: has_year(x, year=2024))
mascara = aqmData["TEM_2024"] | aqmData["STATUS"].apply(is_active)
aqmData = aqmData[mascara].copy()

# 2.5 Injetar 2024 na lista de anos de estações ativas sem o ano registrado
ativas_sem_2024 = aqmData[(aqmData["STATUS"] == "ativa") & (~aqmData["TEM_2024"])].copy()
if not ativas_sem_2024.empty:
    for idx in ativas_sem_2024.index:
        lista_atual = aqmData.at[idx, "ANOS_MONITORADOS"]
        if isinstance(lista_atual, list):
            aqmData.at[idx, "ANOS_MONITORADOS"] = lista_atual + [2024]

# 2.6 Padronizar nomes de categorias de estação
col_target = "CATEGORIA"
if col_target not in aqmData.columns:
    aqmData[col_target] = "Não declarado"
aqmData[col_target] = aqmData[col_target].fillna("Não declarado").astype(str).str.strip().str.capitalize()

correction_map = {
    "Referencia": "Referência", "Referência": "Referência",
    "Indicativa": "Indicativa",
    "Nao declarado": "Não declarado", "Não declarado": "Não declarado",
    "Nao declarada": "Não declarado", "Não declarada": "Não declarado",
}
aqmData[col_target] = aqmData[col_target].replace(correction_map)

# 2.7 Garantir apenas as categorias válidas
valid_cats = ["Referência", "Indicativa", "Não declarado"]
aqmData.loc[~aqmData[col_target].isin(valid_cats), col_target] = "Não declarado"

# 3. Explosão das Linhas por Ano e Agregação

# "Explodir" transforma cada lista de anos em uma linha por ano,
# permitindo contar estações ativas em cada ano individualmente.
df_exploded = aqmData.explode("ANOS_MONITORADOS")
df_exploded = df_exploded.dropna(subset=["ANOS_MONITORADOS"])
df_exploded["ANOS_MONITORADOS"] = df_exploded["ANOS_MONITORADOS"].astype(int)

# 3.1 Filtrar até 2024 e remove duplicatas (caso a injeção tenha criado "2024, 2024")
df_exploded = df_exploded[df_exploded["ANOS_MONITORADOS"] <= 2024]
df_exploded = df_exploded.drop_duplicates(subset=["ID_OEMA", "ANOS_MONITORADOS"])

# 3.2 Contar estações únicas por ano, UF e categoria
df_counts = (
    df_exploded.groupby(["ANOS_MONITORADOS", "UF", "CATEGORIA"])["ID_OEMA"]
    .nunique()
    .reset_index(name="Num_Stations")
)

anos = sorted(df_counts["ANOS_MONITORADOS"].unique())
ufs = sorted(df_counts["UF"].unique())
categorias = ["Referência", "Indicativa", "Não declarado"]


# 4. Paleta de Cores por UF
# Usa o colormap "gist_ncar" do Matplotlib para gerar uma cor distinta para cada UF.
n_colors = len(ufs)
cmap = m_colormaps["gist_ncar"]
sample_points = np.linspace(0, 1, n_colors)
color_sequence = [mcolors.rgb2hex(cmap(i)) for i in sample_points]
color_map = {uf: color_sequence[i] for i, uf in enumerate(ufs)}

# 5. Construção das Traces do Gráfico
# Cada combinação UF x Categoria gera uma trace de barra.
# As listas de visibilidade controlam quais traces ficam visíveis em cada botão.
fig_bars = go.Figure()
visibility_all = []  # Botão "Todas"
visibility_ref = []  # Botão "Referência"
visibility_ind = []  # Botão "Indicativa"
visibility_nd  = []  # Botão "Não Declarado"

for uf in ufs:
    df_uf_total = df_counts[df_counts["UF"] == uf]
    show_legend_uf = True  # Exibe a UF na legenda apenas na primeira trace adicionada

    for cat in categorias:
        df_plot = df_uf_total[df_uf_total["CATEGORIA"] == cat].sort_values("ANOS_MONITORADOS")

        if not df_plot.empty:
            # Melhoria no Hover: Removemos a repetição do nome do Estado e Ano, pois o 'unified' já mostra
            # Mas se preferir ver no box individual, o formato abaixo é limpo:
            custom_hover = (
                f"<b>Tipo: {cat}</b><br>" +
                f"Qtd: %{{y}}" +
                "<extra></extra>" # Remove a caixa lateral padrão do Plotly
            )
            
            fig_bars.add_trace(go.Bar(
                x=df_plot['ANOS_MONITORADOS'],
                y=df_plot['Num_Stations'],
                name=uf, 
                legendgroup=uf, 
                showlegend=show_legend_uf,
                marker_color=color_map[uf],
                customdata=[cat] * len(df_plot),
                hovertemplate=custom_hover
            ))
            
            show_legend_uf = False
            
            # Cada trace recebe True apenas para o botão de sua categoria
            visibility_all.append(True)
            visibility_ref.append(True if cat == "Referência" else False)
            visibility_ind.append(True if cat == "Indicativa" else False)
            visibility_nd.append( True if cat == "Não declarado" else False)


# 6. Botões de Filtro e Layout Final
updatemenus = [dict(
    type="buttons", 
    direction="left",
    pad={"r": 10, "t": 10},
    showactive=True,
    x=0.0,
    xanchor="left",
    y=1.15,
    yanchor="top",
    buttons=list([
            dict(label="Todas", method="update", 
                 args=[{"visible": visibility_all}, {"title": ""}]),
            dict(label="Referência", method="update", 
                 args=[{"visible": visibility_ref}, {"title": ""}]),
            dict(label="Indicativa", method="update", 
                 args=[{"visible": visibility_ind}, {"title": ""}]),
            dict(label="Não Declarado", method="update", 
                 args=[{"visible": visibility_nd}, {"title": ""}]),
        ]),
    )
]

layout_style = dict(
    plot_bgcolor="whitesmoke",
    paper_bgcolor="white",
    hovermode='x unified', # Unifica o hover por Ano
    legend_title="UF",
    height=650,
    updatemenus=updatemenus
)

fig_bars.update_layout(
    **layout_style,
    barmode='stack',
    xaxis_title="Ano",
    yaxis_title="Número de pontos de monitoramento",
    xaxis=dict(type='linear', tickmode='array', tickvals=anos, ticktext=anos)
)


'''  >>> CONFIGURAÇÃO OBRIGATÓRIA <<<
Ajuste output_path para o diretório/arquivo de saída no seu computador antes de executar o script.
Como o código é compartilhado via Git, este caminho varia entre usuários.
Certifique-se de que o diretório exista e que você tenha permissão de escrita. '''

# 7. Exportar HTML, salvar localmente e abrir no navegador
import webbrowser

output_dir = "outputs"
output_path = os.path.join(output_dir, "Figura.20.html")
os.makedirs(output_dir, exist_ok=True)

fig_bars.write_html(output_path, include_plotlyjs="cdn")
webbrowser.open(output_path)

display(HTML(fig_bars.to_html(include_plotlyjs="cdn")))

### Fig. 21 — Acréscimo e decréscimo anual do número de estações de monitoramento por UF
Gráfico de linhas interativo que mostra a **variação anual** (diferença entre anos consecutivos)  
no total de estações de monitoramento por UF.  

Áreas coloridas de fundo destacam crescimento (azul) e redução (rosa).  

Botões permitem filtrar por categoria: **Geral · Referência · Indicativa · Não Declarado**.

In [3]:
# Importações necessárias
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import os
from IPython.display import HTML

# Importa módulos Matplotlib para a paleta de 27 cores
from matplotlib import colormaps as m_colormaps 
from matplotlib import colors as mcolors 

# 1. Caminhos e Leitura do CSV
aqmData = pd.read_csv('https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv')
rootPath = os.path.dirname(os.getcwd())

# 2. FILTRAR SOMENTE ESTAÇÕES QUE MONITORARAM EM 2024
# 2.1 Converter a coluna de anos monitorados de string para lista de inteiros
def parse_anos(x):
    try:
        return [int(z) for z in str(x)
                .replace('[','').replace(']','').replace("'",'')
                .split(',') if z.strip().isdigit()]
    except:
        return []

aqmData['ANOS_MONITORADOS'] = aqmData['ANOS_MONITORADOS'].apply(parse_anos)
aqmData = aqmData[aqmData["UF"] == UF]

# 2.3 Filtro dos dados para apenas as estações que possuem 2024
# Caso queira analisar outro ano, basta alterar o filtro abaixo para o ano desejado.
aqmData = aqmData[aqmData['ANOS_MONITORADOS'].apply(lambda lst: 2024 in lst)]

# 2.4 Remover duplicidades agora garante coerência
aqmData = aqmData.drop_duplicates(subset=['ID_OEMA'])

# 2.5 Padronizar categorias de estação
if 'CATEGORIA' not in aqmData.columns:
    aqmData['CATEGORIA'] = 'Não declarado'

aqmData['CATEGORIA'] = aqmData['CATEGORIA'].fillna('Não declarado').astype(str).str.strip().str.capitalize()
correction_map = {
    'Referencia': 'Referência', 'Indicativa': 'Indicativa',
    'Nao declarado': 'Não declarado', 'Nao declarada': 'Não declarado', 'Não declarada': 'Não declarado'
}
aqmData['CATEGORIA'] = aqmData['CATEGORIA'].replace(correction_map)
valid_cats = ['Referência', 'Indicativa', 'Não declarado']
aqmData.loc[~aqmData['CATEGORIA'].isin(valid_cats), 'CATEGORIA'] = 'Não declarado'

# Tratamento de Anos
if aqmData["ANOS_MONITORADOS"].dtype == object:
    aqmData["ANOS_MONITORADOS"] = aqmData["ANOS_MONITORADOS"].apply(
        lambda x: [int(y) for y in str(x)
                   .replace("[","").replace("]","").replace("'", "")
                   .split(",") if y.strip().isdigit()]
    )

# 3. Explosão e Agregação por Ano
df_exploded = aqmData.explode("ANOS_MONITORADOS")
df_exploded = df_exploded.dropna(subset=['ANOS_MONITORADOS'])
df_exploded['ANOS_MONITORADOS'] = df_exploded['ANOS_MONITORADOS'].astype(int)

# Cria série de totais (todas as categorias juntas) e por categoria individualmente
df_total = (
    df_exploded.groupby(['ANOS_MONITORADOS', 'UF'])['ID_OEMA']
    .nunique().reset_index(name='Num_Stations')
)
df_total['CATEGORIA'] = 'Total'

df_by_cat = (
    df_exploded.groupby(['ANOS_MONITORADOS', 'UF', 'CATEGORIA'])['ID_OEMA']
    .nunique().reset_index(name='Num_Stations')
)
# Combina os dois DataFrames e limita até 2024 ou ano de interesse
# Caso queira analisar outro ano, basta alterar o filtro abaixo para o ano desejado.
df_combined = pd.concat([df_total, df_by_cat])
df_combined = df_combined[df_combined['ANOS_MONITORADOS'] <= 2024]

anos = sorted(df_combined['ANOS_MONITORADOS'].unique())
ufs = sorted(df_combined['UF'].unique())

# Cores 
n_colors = len(ufs)

# OBTENDO O COLORMAP 'gist_ncar' DE FORMA MODERNA
cmap = m_colormaps['gist_ncar'] 

# Gera n_colors pontos linearmente espaçados no colormap
sample_points = np.linspace(0, 1, n_colors) 

# Converte as cores RGB do Matplotlib para o formato Hex (Plotly)
color_sequence = [mcolors.rgb2hex(cmap(i)) for i in sample_points]

# Cria o dicionário UF -> Cor
color_map = {}
for i, uf in enumerate(ufs):
    color_map[uf] = color_sequence[i]


# 4. Construção do Gráfico
# Para cada UF e cada cenário (Total, Referência, Indicativa, Não declarado),
# calcula a diferença anual (delta) e adiciona uma trace de linha ao gráfico.
fig_diff = go.Figure()

# Listas de Controle (Booleanas)
vis_total = []  # Visibilidade para o botão "Geral"
vis_ref   = []  # Visibilidade para o botão "Referência"
vis_ind   = []  # Visibilidade para o botão "Indicativa"
vis_nd    = []  # Visibilidade para o botão "Não Declarado"

all_diffs = []
scenarios = ['Total', 'Referência', 'Indicativa', 'Não declarado']

for uf in ufs:
    for scenario in scenarios:
        # Filtra dados
        mask = (df_combined['UF'] == uf) & (df_combined['CATEGORIA'] == scenario)
        df_uf = df_combined[mask].sort_values('ANOS_MONITORADOS')
        
        # Preenche com 0 para o cálculo de diferença anual
        if df_uf.empty:
            x_data = []
            y_data = []
            diff_list = []
        else:
            df_uf = (
                df_uf.set_index('ANOS_MONITORADOS')['Num_Stations']
                .reindex(anos, fill_value=0)
                .reset_index()
)
            # Calcula a diferença anual
            df_uf['Difference'] = df_uf['Num_Stations'].diff().fillna(0)
            x_data = df_uf['ANOS_MONITORADOS']
            y_data = df_uf['Difference']
            diff_list = df_uf['Difference'].tolist()
        
        all_diffs.extend(diff_list)

        # Define estado INICIAL (apenas Total visível e com legenda)
        is_total = (scenario == 'Total')
        is_ref = (scenario == 'Referência')
        is_ind = (scenario == 'Indicativa')
        is_nd = (scenario == 'Não declarado')

        fig_diff.add_trace(go.Scatter(
            x=x_data,
            y=y_data,
            mode='lines+markers',
            name=uf,
            legendgroup=uf, # Agrupa todos deste estado
            showlegend=is_total, # Inicialmente, legenda exibida apenas na trace "Total" de cada UF
            visible=is_total,    # Inicialmente, só o Total é visível
            line=dict(color=color_map[uf]), 
            marker=dict(color=color_map[uf]),
            hovertemplate=f"<b>{uf}</b> ({scenario})<br>Ano: %{{x}}<br>Diferença: %{{y}}<extra></extra>"
        ))
        
        # Popula as listas para os botões (para visibilidade e legenda)
        vis_total.append(is_total)
        vis_ref.append(is_ref)
        vis_ind.append(is_ind)
        vis_nd.append(is_nd)



# 5. Configuração dos Botões
updatemenus = [
    dict(
        type="buttons",
        direction="left",
        pad={"r": 10, "t": 10},
        showactive=True,
        x=0.0,
        xanchor="left",
        y=1.15,
        yanchor="top",
        buttons=list([
            dict(
                label="Geral", 
                method="update", 
                args=[
                    {"visible": vis_total, "showlegend": vis_total}, 
                    {"title": ""}
                ]
            ),
            dict(
                label="Referência", 
                method="update", 
                args=[
                    {"visible": vis_ref, "showlegend": vis_ref}, 
                    {"title": ""}
                ]
            ),
            dict(
                label="Indicativa", 
                method="update", 
                args=[
                    {"visible": vis_ind, "showlegend": vis_ind}, 
                    {"title": ""}
                ]
            ),
            dict(
                label="Não Declarado", 
                method="update", 
                args=[
                    {"visible": vis_nd, "showlegend": vis_nd}, 
                    {"title": ""}
                ]
            ),

        ]),
    )
]


# 6. Layout e Exportação
# Define range do eixo Y e adiciona retângulos de fundo coloridos para destacar
# regiões de crescimento (azul) e redução (rosa) da rede de monitoramento.
if all_diffs:
    max_val = max(all_diffs)
    #min_val = min(all_diffs)
    min_val=-10
    # Define o range do eixo Y baseado no maior valor absoluto (para centralizar em zero)
    max_abs_value = max(abs(max_val), abs(min_val))
else:
    max_abs_value = 10

layout_style = dict(

    plot_bgcolor="white",
    paper_bgcolor="white",
    hovermode='x unified',
    legend_title="UF",
    height=800,
    updatemenus=updatemenus
)

fig_diff.update_layout(
    **layout_style,
    xaxis_title="Ano",
    yaxis_title="Diferença anual",
    xaxis_range=[min(anos) - 0.5, 2024.5],
    xaxis=dict(
        type='linear',
        title="Ano",
        tickmode='array',
        tickvals=anos,
        ticktext=anos
    ),
    yaxis_range=[min_val, max_abs_value * 1.1], # Aplica o range centralizado
    shapes=[
        # Zona de redução (abaixo de zero)
        dict(type="rect", xref="paper", x0=0, x1=1, yref="y", y0=-max_abs_value * 1.1, y1=0,  
             fillcolor="mistyrose", opacity=0.5, layer="below", line_width=0),
        # Zona de crescimento (acima de zero)
        dict(type="rect", xref="paper", x0=0, x1=1, yref="y", y0=max_abs_value * 1.1, y1=0,  
             fillcolor="lightblue", opacity=0.5, layer="below", line_width=0),
        # Linha zero central
        dict(type="line", xref="paper", x0=0, x1=1, yref="y", y0=0, y1=0,
             line=dict(color="gray", width=2), layer="below")
    ]
)

'''  >>> CONFIGURAÇÃO OBRIGATÓRIA <<<
Ajuste output_path para o diretório/arquivo de saída no seu computador antes de executar o script.
Como o código é compartilhado via Git, este caminho varia entre usuários.
Certifique-se de que o diretório exista e que você tenha permissão de escrita. '''

# 7. Exportar HTML, salvar localmente e abrir no navegador
import webbrowser

output_dir = "outputs"
output_path = os.path.join(output_dir, "Figura.21.html")
os.makedirs(output_dir, exist_ok=True)

fig_diff.write_html(output_path, include_plotlyjs="cdn")
webbrowser.open(output_path)

display(HTML(fig_diff.to_html(include_plotlyjs="cdn")))

### Fig. 22 — Número de pontos de monitoramento por poluente e UF

Gráfico de barras empilhadas interativo que mostra, para cada poluente atmosférico selecionado,  
a evolução anual do número de pontos de monitoramento por UF.  

Um dropdown permite selecionar o poluente desejado (padrão: MP25).  
Um segundo dropdown permite filtrar por categoria de estação.  

**Nota:** apenas pontos de monitoramento com datas de início e fim de operação registradas foram incluídos nesta figura.

In [4]:
# Importações necessárias
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import HTML


# 1. Carregar e preparar dados
df = pd.read_csv('https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv')
rootPath = os.path.dirname(os.getcwd())

# Normalizar UF: remover espaços acidentais
df["UF"] = df["UF"].astype(str).str.replace(" ", "")
df = df[df["UF"] == UF]

# Padronizar nomenclatura: PM25 -> MP25 (convencao adotada no relatorio)
df.loc[df["POLUENTE"] == "PM25", "POLUENTE"] = "MP25"

# Tratamento de categorias
df["CATEGORIA"] = (
    df.get("CATEGORIA", "Não declarado")
      .fillna("Não declarado")
      .astype(str)
      .str.strip()
      .str.capitalize()
)

map_cat = {
    "Referencia": "Referência",
    "Indicativa": "Indicativa",
    "Nao declarado": "Não declarado",
    "Nao declarada": "Não declarado",
    "Não declarada": "Não declarado",
}
df["CATEGORIA"] = df["CATEGORIA"].replace(map_cat)

valid = ["Referência", "Indicativa", "Não declarado"]
df.loc[~df["CATEGORIA"].isin(valid), "CATEGORIA"] = "Não declarado"


# 2. Explosao por Ano e Filtro ate 2024
# A coluna ANOS_MONITORADOS contem listas de anos como string (ex.: "2020,2021,2022").
# str.split(",") converte para lista; explode() cria uma linha por ano por estacao.
df["ANOS_MONITORADOS"] = df["ANOS_MONITORADOS"].astype(str).str.split(",")
df = df.explode("ANOS_MONITORADOS")

# Descartar valores nao numericos gerados por celulas vazias ou mal formatadas
df = df[pd.to_numeric(df["ANOS_MONITORADOS"], errors="coerce").notna()]
df["ANOS_MONITORADOS"] = df["ANOS_MONITORADOS"].astype(int)

# Se quiser, limitar o historico ao ano de referencia do relatorio
df = df[df["ANOS_MONITORADOS"] <= 2024]


# 3. Agregacao por Poluente / UF / Categoria / Ano
# .size() conta quantas linhas (= pontos de monitoramento) existem para cada grupo.
# O resultado e o numero de pontos ativos por poluente, UF, categoria e ano.
df_ag = (
    df.groupby(["POLUENTE", "UF", "CATEGORIA", "ANOS_MONITORADOS"])
      .size()
      .reset_index(name="NSTATION")
)

# Renomear coluna de ano pra facilitar no JS
df_ag = df_ag.rename(columns={"ANOS_MONITORADOS": "ANO"})


# 4. Cores por UF (MESMAS DO ALTAIR)
# Usa o colormap "gist_ncar" do Matplotlib para garantir cores distintas
# mesmo com ~27 UFs. np.linspace distribui os pontos uniformemente no colormap.
ufs = sorted(df_ag["UF"].unique())
n_colors = len(ufs)

cmap = plt.get_cmap("gist_ncar")
sample_points = np.linspace(0, 1, n_colors)
color_sequence = [mcolors.to_hex(cmap(i)) for i in sample_points]

# Dicionario UF -> cor hex, compartilhado entre Python e o JavaScript embutido no HTML
uf_colors = {uf: color_sequence[i] for i, uf in enumerate(ufs)}


# 5. Dados → JSON para o JS
# Os dados sao embutidos diretamente no HTML como variaveis JavaScript.
# Isso torna o arquivo HTML completamente autossuficiente e interativo
# sem necessidade de servidor Python ou Jupyter em execucao.
rows = df_ag.to_dict(orient="records")
json_data = json.dumps(rows, ensure_ascii=False)
uf_colors_json = json.dumps(uf_colors, ensure_ascii=False)


# 6. Template HTML + JS (Plotly)
# A figura e construida como HTML puro usando Plotly.js carregado via CDN.
# A logica de filtragem e renderizacao reside inteiramente no JavaScript:
#   - updatePlot(): filtra DATA pelos dropdowns e chama Plotly.newPlot()
#   - Os dropdowns sao populados dinamicamente com os valores unicos do dataset

html_template = r"""
<!DOCTYPE html>
<html>
<head>
<meta charset="utf-8"/>
<script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
<style>
body {
  font-family: Arial, sans-serif;
  margin: 0;
  padding: 14px;
  background: #fff;
}
#controls {
  display: flex;
  flex-wrap: wrap;
  gap: 16px;
  align-items: center;
  margin-bottom: 10px;
}
.control {
  display: flex;
  align-items: center;
  gap: 6px;
}
.label {
  font-size: 14px;
  font-weight: 600;
  color: #333;
  white-space: nowrap;
}
.select {
  appearance: none;
  padding: 4px 8px;
  border: 1px solid #999;
  border-radius: 4px;
  background: #f9f9f9;
  cursor: pointer;
  font-size: 13px;
}
#figMonitor {
  width: 100%;
  height: 650px;
}
</style>
</head>

<body>

<div id="controls">
  <div class="control">
    <span class="label">Poluente:</span>
    <select id="selPol" class="select"></select>
  </div>

  <div class="control">
    <span class="label">Categoria:</span>
    <select id="selCat" class="select"></select>
  </div>
</div>

<div id="figMonitor"></div>

<script>
const DATA = __DATA__;
const UF_COLORS = __UF_COLORS__;

// ===== Preenche dropdowns =====
const selPol = document.getElementById("selPol");
const selCat = document.getElementById("selCat");

function uniqueSorted(arr) {
  return Array.from(new Set(arr)).sort();
}

// Poluentes
const poluentes = uniqueSorted(DATA.map(d => d.POLUENTE));
poluentes.forEach(p => {
  const opt = document.createElement("option");
  opt.value = p;
  opt.textContent = p;
  selPol.appendChild(opt);
});

// Categorias
const categorias = uniqueSorted(DATA.map(d => d.CATEGORIA));
selCat.appendChild(new Option("Todas", "Todas"));
categorias.forEach(c => {
  const opt = document.createElement("option");
  opt.value = c;
  opt.textContent = c;
  selCat.appendChild(opt);
});

// Valores padrão
if (poluentes.includes("MP25")) {
  selPol.value = "MP25";
}

// ===== Atualizar gráfico =====
function updatePlot() {
  const pol = selPol.value;
  const cat = selCat.value;

  // Filtrar dados
  const filtered = DATA.filter(d =>
    d.POLUENTE === pol &&
    (cat === "Todas" || d.CATEGORIA === cat)
  );

  if (filtered.length === 0) {
    Plotly.newPlot("figMonitor", [], {
      title: {text: "Sem dados para o filtro selecionado", x: 0.5}
    });
    return;
  }

  // Anos e UFs
  const anos = uniqueSorted(filtered.map(d => d.ANO)).map(Number);
  const ufs = uniqueSorted(filtered.map(d => d.UF));

  const traces = ufs.map(uf => {
    const ys = anos.map(ano => {
      return filtered
        .filter(d => d.UF === uf && Number(d.ANO) === ano)
        .reduce((acc, d) => acc + Number(d.NSTATION || 0), 0);
    });
    return {
      type: "bar",
      name: uf,
      x: anos,
      y: ys,
      marker: { color: UF_COLORS[uf] || "#888888" },
      hovertemplate: "UF: " + uf + "<br>Ano: %{x}<br>N. pontos: %{y}<extra></extra>"
    };
  });

  const layout = {
    barmode: "stack",
    template: "plotly_white",
    title: {
      text: "",
      x: 0.5
    },
    xaxis: {
      title: "Ano",
      dtick: 1
    },
    yaxis: {
      title: "N. pontos de monitoramento"
    },
    legend: {
      orientation: "h",
      yanchor: "bottom",
      y: -0.25,
      xanchor: "left",
      x: 0,
      title: { text: "UF" }
    },
    margin: { l: 60, r: 20, t: 60, b: 80 },
    hovermode: "x unified",
    hoverlabel: {
      bgcolor: "white",
      bordercolor: "#444",
      font: { size: 11 }
    }
  };

  Plotly.newPlot("figMonitor", traces, layout, {displayModeBar: false});
}

// Eventos
selPol.addEventListener("change", updatePlot);
selCat.addEventListener("change", updatePlot);

// Plot inicial
updatePlot();
</script>

</body>
</html>
"""

# Substitui os placeholders pelo JSON de dados e cores
html_code = (
    html_template
    .replace("__DATA__", json_data)
    .replace("__UF_COLORS__", uf_colors_json)
)

HTML(html_code)


In [5]:
# Para salvar a imagem e abrir no navegador, ajuste o caminho de saída conforme necessário.

'''  >>> CONFIGURAÇÃO OBRIGATÓRIA <<<
Ajuste output_path para o diretório/arquivo de saída no seu computador antes de executar o script.
Como o código é compartilhado via Git, este caminho varia entre usuários.
Certifique-se de que o diretório exista e que você tenha permissão de escrita. '''

import webbrowser

# Salvar figura interativa localmente e abrir no navegador
output_dir = "outputs"
output_path = os.path.join(output_dir, "Figura.22.html")

os.makedirs(output_dir, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(html_code)

webbrowser.open(output_path)

False